# Data Cleaning - Quantium Chips

## 1. Imports

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

## 2. Data Loading

In [ ]:
transaction = pd.read_excel('/Users/thibautraiola/DATA-ANALYST/Project4Portfolio/Forage/Quantium - DA /1. Data preparation & customer analytics/data/QVI_transaction_data.xlsx')
pur_behavior = pd.read_csv('/Users/thibautraiola/DATA-ANALYST/Project4Portfolio/Forage/Quantium - DA /1. Data preparation & customer analytics/data/QVI_purchase_behaviour.csv')

## 3. Exploration - transaction table

In [ ]:
transaction.head()

In [ ]:
transaction.info()

.describe() for numeric values: STORE_NBR, PROD_NBR, PROD_QTY, TOT_SALES

In [ ]:
transaction['TOT_SALES'].describe()

.unique() for text values

In [ ]:
transaction['PROD_NAME'].unique()

Checking PROD_NAME sorted (extra spaces, characters, non-chips products)

In [ ]:
sorted(transaction['PROD_NAME'].unique())

## 4. Cleaning - transaction table

In [ ]:
transaction['DATE'] = pd.to_datetime(transaction['DATE'], origin='1899-12-30', unit='D')

In [ ]:
transaction['PACK_SIZE'] = transaction['PROD_NAME'].str.extract(r'(\d{2,3})[gG]').astype(int)

In [ ]:
transaction['BRAND_NAME'] = transaction['PROD_NAME'].str.split().str[0]
transaction['BRAND_NAME'] = transaction['BRAND_NAME'].replace({'Smith':'Smiths', 'WW':'Woolworths','NCC':'Natural', 'Dorito':'Doritos', 'GrnWves':'Grain', 'Infzns':'Infuzions','RRD':'Red','Snbts':'Sunbites'})
transaction['BRAND_NAME'].unique()

In [ ]:
transaction[['PROD_NBR', 'PROD_NAME']].drop_duplicates()
transaction[transaction['PROD_NAME'].str.contains('Dip|Salsa')][['PROD_NBR', 'PROD_NAME']].drop_duplicates()

In [ ]:
dips_salsa = [57,107,101,65,35,59,76,41]

In [ ]:
transaction = transaction[~transaction['PROD_NBR'].isin(dips_salsa)]

In [ ]:
transaction['PROD_NAME'].unique()

Checking for duplicates

In [ ]:
transaction.duplicated().sum()

In [ ]:
transaction[transaction.duplicated(keep=False)]

In [ ]:
transaction = transaction.drop_duplicates()

## 5. Exploration - pur_behavior table

In [ ]:
pur_behavior.info()

In [ ]:
pur_behavior['LIFESTAGE'].unique()

In [ ]:
pur_behavior['PREMIUM_CUSTOMER'].unique()

In [ ]:
pur_behavior['LYLTY_CARD_NBR'].duplicated().sum()

## 6. Merging the tables

In [ ]:
merged = pd.merge(transaction, pur_behavior, on='LYLTY_CARD_NBR', how='left')

In [ ]:
merged.info()

In [ ]:
merged.describe()

## 7. Post-merge checks

Checking for missing values after the merge (unmatched LYLTY_CARD_NBR), dates with no transaction

In [ ]:
merged.isna().sum()

In [ ]:
all_dates = pd.date_range(start=merged['DATE'].min(), end=merged['DATE'].max(), freq='D')
missing_dates = all_dates.difference(merged['DATE'])
missing_dates


## 8. Outlier detection

In [ ]:
sns.scatterplot(data=merged,y='TOT_SALES',x='PROD_QTY');

In [ ]:
merged[(merged['TOT_SALES']==650) | (merged['PROD_QTY']==200)]

In [ ]:
merged = merged[merged['LYLTY_CARD_NBR']!=226000]

## 9. Dataset saving

In [ ]:
merged.to_csv('/Users/thibautraiola/DATA-ANALYST/Project4Portfolio/Forage/Quantium - DA /1. Data preparation & customer analytics/cleaned_data.csv', index=False)

In [ ]:
merged